# Personality Type Prediction – Part 2: Evaluation and Tuning

In this notebook I check the model in a better way (classification report, cross-validation, F1-macro).
Then I try out different settings for the Random Forest.

## 1. Setup

Same data cleaning and pipeline as in notebook 01.

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# load the data
df = pd.read_csv("data.csv")

# data cleaning (same as notebook 01)
item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]
df.loc[df["age"] > 100, "age"] = np.nan
df[item_cols] = df[item_cols].replace(0, np.nan)
df = df.drop_duplicates()

# features and target
X = df.drop(columns=["target"])
y = df["target"]

# preprocessing pipeline
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), item_cols + ["age"]),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), ["gender", "hand"]),
])

print("Shape:", df.shape)

Shape: (19716, 23)


## 2. Classification report

Accuracy alone can be misleading because the classes are not balanced.
The classification report shows precision, recall and F1 for each class.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))

                 precision    recall  f1-score   support

       Moderate       0.78      0.93      0.85      1690
 Overcontroller       0.88      0.85      0.87       566
      Resilient       0.94      0.96      0.95      1233
Undercontroller       0.73      0.21      0.32       455

       accuracy                           0.84      3944
      macro avg       0.83      0.74      0.75      3944
   weighted avg       0.84      0.84      0.82      3944



**Notes:**
- Accuracy is 0.84, but F1-macro is only 0.75.
- Undercontroller is the hardest class (F1 = 0.32). The model finds only 21% of them (recall).
- I think most of them are predicted as Moderate, because Moderate is the biggest class.

## 3. Cross-validation

One train/test split can be lucky or unlucky.
With cross-validation the data is split 5 times, so every row is used for testing once.

In [4]:
from sklearn.model_selection import cross_val_score

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

scores = cross_val_score(pipe, X, y, cv=5, scoring="f1_macro")
print("Fold scores:", scores.round(4))
print("Average F1-macro:", scores.mean().round(4))

Fold scores: [0.7477 0.7404 0.7532 0.7517 0.7397]
Average F1-macro: 0.7465


**Notes:**
- The 5 fold scores are between 0.740 and 0.753, so the model is stable.
- The average F1-macro is 0.7465. This is very close to the single split (0.75).
- Cross-validation is more trustworthy than one split, because it uses 5 different test sets.

## 4. Try different settings (max_depth)

A hyperparameter is a setting that I choose before training.
Here I try different values for max_depth and compare the F1-macro.

In [5]:
for depth in [3, 5, 10, 20, None]:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(max_depth=depth, random_state=42, n_jobs=-1)),
    ])
    score = cross_val_score(pipe, X, y, cv=5, scoring="f1_macro").mean()
    print(f"max_depth={depth}: F1-macro = {score:.4f}")

max_depth=3: F1-macro = 0.5754
max_depth=5: F1-macro = 0.6188
max_depth=10: F1-macro = 0.7119
max_depth=20: F1-macro = 0.7482
max_depth=None: F1-macro = 0.7465


**Notes:**
- With small max_depth (3, 5) the score is much lower. The trees are too simple.
- From max_depth=20 the score does not get better anymore.
- 20 (0.7482) and None (0.7465) are almost the same. The difference is smaller than the difference between the folds.